<a href="https://www.kaggle.com/code/avikdas567/rental-product-recommendation-system?scriptVersionId=295900252" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import gc
import warnings
import re

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION
# ==========================================
BASE_PATH = '/kaggle/input/rental-product-recommendation-system/'
FILES = {
    'hits': BASE_PATH + 'metrika_hits_test.csv',
    'visits': BASE_PATH + 'metrika_visits_test.csv',
    'products': BASE_PATH + 'new_site_products.csv',
    'orders': BASE_PATH + 'new_site_orders.csv'
}

# ==========================================
# 1. LOAD PRODUCT CATALOG
# ==========================================
print("Loading Product Catalog...")
products_df = pd.read_csv(FILES['products'], usecols=['id', 'slug'])
slug_to_id = dict(zip(products_df['slug'], products_df['id']))
del products_df
gc.collect()

# ==========================================
# 2. BUILD CO-PURCHASE MATRIX (Orders)
# ==========================================
print("Building Co-Purchase Matrix (Orders)...")
orders_df = pd.read_csv(FILES['orders'], usecols=['product_id', 'number'])

# 1. Global Popularity (Fallback)
top_products = orders_df['product_id'].value_counts().head(6).index.tolist()

# 2. Co-occurrence Matrix
# We use a nested dict: matrix[item_a][item_b] = score
order_matrix = defaultdict(lambda: defaultdict(float))

# Group by Order Number
order_groups = orders_df.groupby('number')['product_id'].apply(list)

for items in order_groups:
    if len(items) > 1:
        # Weight formula: 1 / (order_size - 1)
        # This prevents massive bulk orders from dominating the stats
        w = 1.0 / (len(items) - 1)
        for i in items:
            for j in items:
                if i != j:
                    order_matrix[i][j] += w

print(f"Processed {len(order_groups)} orders.")
del orders_df, order_groups
gc.collect()

# ==========================================
# 3. BUILD CO-VISITATION MATRIX (Sessions)
# ==========================================
print("Building Co-Visitation Matrix (Test Sessions)...")

# A. Load Hits & Map to IDs
hits_df = pd.read_csv(FILES['hits'], usecols=['watch_id', 'slug'], dtype={'watch_id': str})
hits_df['product_id'] = hits_df['slug'].map(slug_to_id)
hits_df = hits_df.dropna(subset=['product_id'])

# Create Lookup: watch_id -> product_id
watch_id_to_product = dict(zip(hits_df['watch_id'], hits_df['product_id'].astype(int)))
del hits_df
gc.collect()

# B. Process Sessions
visits_df = pd.read_csv(FILES['visits'], usecols=['visit_id', 'watch_ids'])
view_matrix = defaultdict(lambda: defaultdict(float))
session_histories = {}
pattern = re.compile(r'\d+') 

# Tuneable Parameters
WEIGHT_ORDER = 5.0  # Buying is 5x stronger than viewing
WEIGHT_VIEW = 1.0   # Viewing is baseline

print("Processing sessions...")
for row in visits_df.itertuples(index=False):
    visit_id = row[0]
    raw_ids = pattern.findall(str(row[1]))
    
    # Resolve IDs
    history = []
    for wid in raw_ids:
        if wid in watch_id_to_product:
            history.append(watch_id_to_product[wid])
    
    session_histories[visit_id] = history
    
    # Update View Matrix (Weighted by proximity)
    # If users look at A then B, they are related.
    if len(history) > 1:
        for i in range(len(history)):
            item_a = history[i]
            
            # Look at items viewed shortly before or after (Window +/- 3)
            start = max(0, i-3)
            end = min(len(history), i+4)
            
            for j in range(start, end):
                if i != j:
                    item_b = history[j]
                    
                    # Distance decay: 
                    # Adjacent (dist=1) -> weight=1.0
                    # Dist=2 -> weight=0.5
                    # Dist=3 -> weight=0.33
                    diff = abs(i - j)
                    w = 1.0 / diff
                    
                    view_matrix[item_a][item_b] += w

# ==========================================
# 4. GENERATE PREDICTIONS
# ==========================================
print("Generating Predictions...")
final_preds = []

# Pre-calculate global best string
global_fill = top_products[:6]

for visit_id in visits_df['visit_id']:
    history = session_histories.get(visit_id, [])
    scores = defaultdict(float)
    seen_items = set(history)
    
    if history:
        # STRATEGY: Weighted Recency
        # We give the Last Item much more weight than the 2nd to last
        # Weights: [..., 0.33, 0.5, 1.0] (Last item gets 1.0)
        
        # Consider last 5 items
        recent_items = history[-5:]
        n = len(recent_items)
        
        for i, item in enumerate(recent_items):
            # Calculate position weight (Linear increase towards end)
            # Example for 3 items: 0.33, 0.66, 1.0
            pos_weight = (i + 1) / n
            
            # 1. Add votes from ORDER Matrix (High Confidence)
            if item in order_matrix:
                for cand, w in order_matrix[item].items():
                    if cand not in seen_items:
                        scores[cand] += (w * WEIGHT_ORDER) * pos_weight
            
            # 2. Add votes from VIEW Matrix (Discovery)
            if item in view_matrix:
                for cand, w in view_matrix[item].items():
                    if cand not in seen_items:
                        scores[cand] += (w * WEIGHT_VIEW) * pos_weight

    # Select Top 6 Candidates
    if scores:
        best_candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        prediction = [x[0] for x in best_candidates[:6]]
    else:
        prediction = []
    
    # FILLER 1: If we have < 6, fill with Global Bestsellers
    if len(prediction) < 6:
        for item in global_fill:
            if item not in prediction and item not in seen_items:
                prediction.append(item)
            if len(prediction) == 6:
                break
                
    # FILLER 2: Final safety net (duplicates allowed if absolutely necessary to reach 6, strictly distinct preferred)
    if len(prediction) < 6:
        for item in global_fill:
            if item not in prediction:
                prediction.append(item)
            if len(prediction) == 6:
                break

    final_preds.append(" ".join(map(str, prediction)))

# ==========================================
# 5. SUBMIT
# ==========================================
visits_df['product_ids'] = final_preds
submission = visits_df[['visit_id', 'product_ids']]
submission.to_csv('submission.csv', index=False)

print("Success! Submission created.")
display(submission.head())

Loading Product Catalog...
Building Co-Purchase Matrix (Orders)...
Processed 2259 orders.
Building Co-Visitation Matrix (Test Sessions)...
Processing sessions...
Generating Predictions...
Success! Submission created.


,visit_id,product_ids
0,6034151852304141635,463480429 463480693 495400618 463480694 495401...
1,0473312698303184850,463480429 463480693 495400618 463480694 495401...
2,0140437049255950634,463480255 463480256 463480466 463480236 463480...
3,4595976301611479438,463480429 463480693 495400618 463480694 495401...
4,1835293676913553579,495277589 463480429 463480225 463480224 463480...
